# Green SM Foundation — full corpus pretraining

This notebook trains the **linked** Green SM Foundation checkpoint on the immutable full v1 corpus. The paired-control Foundation root is mounted and validated as part of the preflight, but it is not trained in this run. Use a distinct run ID (for example `full-v1-r2`) for a genuinely new run; do not delete or overwrite an existing run directory.

In [ ]:
!git clone --branch feat/foundation-encoder \
  https://github.com/huongiang171/green-sm-foundation-pretraining.git \
  /kaggle/working/repo

%cd /kaggle/working/repo
!git log --oneline -5
!pip install -q -e .

In [ ]:
from pathlib import Path
import json
import torch

# The exact Kaggle input path supplied for the control Foundation root.
CONTROL_ROOT = Path(
    '/kaggle/input/datasets/huongiang/green-sm-sim-full/'
    'green_sm_sim_full_v1/green_sm_sim_control_paired_v1/foundation'
)

# Kaggle can shorten the mount path. Fall back to the unique corpus directory.
if not CONTROL_ROOT.is_dir():
    matches = list(Path('/kaggle/input').rglob('green_sm_sim_control_paired_v1'))
    assert len(matches) == 1, f'Expected one control corpus directory, found: {matches}'
    CONTROL_ROOT = matches[0] / 'foundation'

CORPUS_ROOT = CONTROL_ROOT.parent.parent
LINKED_ROOT = CORPUS_ROOT / 'green_sm_sim_linked_v1' / 'foundation'

for required in (
    LINKED_ROOT / 'manifest.json',
    CONTROL_ROOT / 'manifest.json',
    LINKED_ROOT.parent / 'reports' / 'data_quality_report.json',
    CONTROL_ROOT.parent / 'reports' / 'data_quality_report.json',
):
    assert required.is_file(), f'Missing required corpus artifact: {required}'

linked_report = json.loads((LINKED_ROOT.parent / 'reports' / 'data_quality_report.json').read_text())
control_report = json.loads((CONTROL_ROOT.parent / 'reports' / 'data_quality_report.json').read_text())
assert linked_report['readiness_passed'] is True
assert control_report['readiness_passed'] is True
assert linked_report['snapshot_counts'] == {'train': 96000, 'validation': 24000, 'test': 24000}
assert linked_report['target_world_fingerprint'] == control_report['target_world_fingerprint']
assert torch.cuda.is_available(), 'Enable a Kaggle GPU Accelerator before training.'

print('GPU:', torch.cuda.get_device_name(0))
print('Linked Foundation root:', LINKED_ROOT)
print('Control Foundation root:', CONTROL_ROOT)
print('Target-world fingerprint:', linked_report['target_world_fingerprint'])

In [ ]:
import json
from pathlib import Path

REPO = Path('/kaggle/working/repo')
RUN_ID = 'full-v1-r1'
OUTPUT_DIR = Path('/kaggle/working/artifacts/green_sm_pretraining_full')
RUNTIME_CONFIG = REPO / 'configs' / 'pretrain_green_sm_full_kaggle.json'

config = json.loads((REPO / 'configs' / 'pretrain_green_sm_full.json').read_text())
config['linked_root'] = str(LINKED_ROOT)
config['control_root'] = str(CONTROL_ROOT)
config['output_dir'] = str(OUTPUT_DIR)
config['run_id'] = RUN_ID

assert config['training']['max_steps'] == 20000
assert config['model']['max_length'] == config['dataset']['max_tokens'] == 256
run_dir = OUTPUT_DIR / RUN_ID / 'linked'
assert not run_dir.exists(), (
    f'Run directory already exists: {run_dir}. '
    'Choose a new RUN_ID for a fresh run; never overwrite an old run.'
)

RUNTIME_CONFIG.write_text(json.dumps(config, indent=2) + '\n')
print('Runtime config:', RUNTIME_CONFIG)
print('Run directory:', run_dir)
print('Configured training steps:', config['training']['max_steps'])

In [ ]:
%cd /kaggle/working/repo

# Read-only full-corpus/runtime gate. This creates no run artifacts.
from src.pretraining.config import load_pretraining_config
from src.pretraining.gates import preflight_green_sm

preflight = preflight_green_sm(load_pretraining_config('configs/pretrain_green_sm_full_kaggle.json'))
print(preflight)

In [ ]:
%cd /kaggle/working/repo

# Full linked-corpus pretraining: 20,000 optimizer steps.
# Do not add --resume here. A later continuation must use a verified,
# compatible last.pt in this exact run directory.
!python -m src.pretraining \
  --config configs/pretrain_green_sm_full_kaggle.json \
  --corpus linked \
  --device cuda

In [ ]:
from pathlib import Path
import json

run = OUTPUT_DIR / RUN_ID / 'linked'
for name in (
    'vocabulary.json',
    'loss_trace.json',
    'last.pt',
    'last_manifest.json',
    'best.pt',
    'best_manifest.json',
    'checkpoint_manifest.json',
    'run_manifest.json',
):
    assert (run / name).is_file(), f'Missing artifact: {run / name}'

manifest = json.loads((run / 'run_manifest.json').read_text())
checkpoint = json.loads((run / 'checkpoint_manifest.json').read_text())
assert manifest['effective_step_count'] == 20000
assert checkpoint['selected_kind'] == 'best'
print('Completed steps:', manifest['effective_step_count'])
print('Evaluation points:', len(manifest['evaluations']))
print('Selected checkpoint:', checkpoint['checkpoint'])
print('Selected checkpoint SHA-256:', checkpoint['checkpoint_sha256'])

In [ ]:
import json
import matplotlib.pyplot as plt

losses = json.loads((run / 'loss_trace.json').read_text())
evaluations = json.loads((run / 'run_manifest.json').read_text())['evaluations']

plt.figure(figsize=(12, 4))
plt.plot([row['step'] for row in losses], [row['total'] for row in losses], alpha=0.35, label='train step loss')
plt.plot([row['step'] for row in evaluations], [row['validation_total'] for row in evaluations], marker='o', label='validation loss')
plt.xlabel('optimizer step')
plt.ylabel('pretraining loss')
plt.title('Green SM Foundation — full-v1 linked corpus')
plt.grid(alpha=0.25)
plt.legend()
plt.show()

best = min(evaluations, key=lambda row: row['validation_total'])
print('Best validation point:', best)

In [ ]:
from src.training.checkpoint import load_foundation_checkpoint

embedder = load_foundation_checkpoint(run / 'best.pt', run / 'best_manifest.json')
assert not any(parameter.requires_grad for parameter in embedder.parameters())
print(type(embedder).__name__)
print('Foundation checkpoint verified and frozen for downstream use.')

In [ ]:
import shutil

export_dir = Path('/kaggle/working/green_sm_full_v1_checkpoint')
archive_base = Path('/kaggle/working/green_sm_full_v1_checkpoint')
assert not export_dir.exists(), f'Refusing to overwrite existing export: {export_dir}'
shutil.copytree(run, export_dir)
archive = shutil.make_archive(str(archive_base), 'zip', export_dir.parent, export_dir.name)
print('Checkpoint directory:', export_dir)
print('Checkpoint archive:', archive)